# AgentOpt V2 Notebook

In [ ]:
# AgentOpt V2 Notebook
!pip install -q openai tavily-python rich tabulate

import itertools, time
from google.colab import userdata
from openai import OpenAI
from tavily import TavilyClient
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from IPython.display import Markdown, display

console = Console(width=120)

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)
tavily = TavilyClient(api_key=TAVILY_API_KEY)

QUESTION = "Compare AWS and Azure for GenAI workloads."
MODELS = ["gpt-4o-mini","gpt-4o"]
MODEL_COST = {"gpt-4o-mini":1,"gpt-4o":5}

def call_llm(model,prompt):
    s=time.time()
    r=client.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":prompt}]
    )
    return r.choices[0].message.content, r.usage.total_tokens, time.time()-s

def search(q):
    res=tavily.search(query=q,max_results=3)
    return "\n\n".join([x["content"][:600] for x in res["results"]])

def pipeline(question, planner, summarizer):
    plan, ptok, plat = call_llm(planner, f"Create a research plan for:\n{question}")
    evidence = search(question)
    ans, stok, slat = call_llm(summarizer, f"Question:{question}\nPlan:{plan}\nEvidence:{evidence}\nAnswer.")
    return {
        "planner": planner,
        "summarizer": summarizer,
        "answer": ans,
        "tokens": ptok+stok,
        "latency": plat+slat,
        "cost": MODEL_COST[planner]+MODEL_COST[summarizer]
    }

def judge(q,a):
    score,_,_=call_llm("gpt-4o-mini",f"Score 1-10 only.\nQuestion:{q}\nAnswer:{a}")
    try:
        return max(1,min(10,int(''.join(filter(str.isdigit,score))[:2])))
    except:
        return 5

results=[]
for planner,summarizer in itertools.product(MODELS,MODELS):
    console.print(f"[bold cyan]Running {planner} -> {summarizer}[/bold cyan]")
    r=pipeline(QUESTION, planner, summarizer)
    r["quality"]=judge(QUESTION,r["answer"])
    r["score"]=0.8*r["quality"]-0.05*r["latency"]-0.1*r["cost"]
    results.append(r)

results=sorted(results,key=lambda x:x["score"], reverse=True)

tbl=Table(title="AgentOpt Search Results")
for c in ["Planner","Summarizer","Quality","Latency","Tokens","Cost","Score"]:
    tbl.add_column(c)

for r in results:
    tbl.add_row(
        r["planner"], r["summarizer"], str(r["quality"]),
        f"{r['latency']:.2f}s", str(r["tokens"]),
        str(r["cost"]), f"{r['score']:.2f}"
    )
console.print(tbl)

for r in results:
    console.print(Panel.fit(
        f"Quality: {r['quality']}/10\nLatency: {r['latency']:.2f}s\nTokens: {r['tokens']}\nScore: {r['score']:.2f}",
        title=f"{r['planner']} → {r['summarizer']}"
    ))
    display(Markdown(r["answer"]))

winner=results[0]
console.print(Panel.fit(
f"""Planner: {winner['planner']}
Summarizer: {winner['summarizer']}
Quality: {winner['quality']}
Latency: {winner['latency']:.2f}s
Score: {winner['score']:.2f}""",
title="🏆 Best Configuration"))


Running gpt-4o-mini -> gpt-4o-mini

Running gpt-4o-mini -> gpt-4o

Running gpt-4o -> gpt-4o-mini

Running gpt-4o -> gpt-4o

                         AgentOpt Search Results                         
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃ Planner     ┃ Summarizer  ┃ Quality ┃ Latency ┃ Tokens ┃ Cost ┃ Score ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ gpt-4o-mini │ gpt-4o-mini │ 10      │ 28.38s  │ 3270   │ 2    │ 6.38  │
│ gpt-4o      │ gpt-4o      │ 10      │ 13.36s  │ 2963   │ 10   │ 6.33  │
│ gpt-4o      │ gpt-4o-mini │ 9       │ 27.11s  │ 3549   │ 6    │ 5.24  │
│ gpt-4o-mini │ gpt-4o      │ 8       │ 21.32s  │ 2943   │ 6    │ 4.73  │
└─────────────┴─────────────┴─────────┴─────────┴────────┴──────┴───────┘

...

╭─ gpt-4o → gpt-4o-mini ─╮
│ Quality: 9/10          │
│ Latency: 27.11s        │
│ Tokens: 3549           │
│ Score: 5.24            │
╰────────────────────────╯

### A Comparative Analysis of AWS and Azure for Generative AI Workloads

**Introduction:**
In recent years, Generative AI (GenAI) has gained significant traction across various sectors due to its transformative potential. Deploying GenAI workloads efficiently requires robust cloud platforms that can provide the necessary computational power, tools, and services. This analysis will evaluate AWS (Amazon Web Services) and Azure (Microsoft Azure) to determine their strengths and weaknesses in managing GenAI workloads.

#### Research Objectives Summary:
1. **Evaluating Capabilities**: Understand AWS and Azure's capabilities for GenAI workloads.
2. **Feature Comparison**: Compare key features, performance, cost, and usability for GenAI applications on both platforms.
3. **Scalability and Security**: Examine how both platforms handle scalability, security, and ecosystem support.

---

### Phase 1: Literature Review and Background Study

**Key Findings:**
- **Generative AI Technologies**: Fundamental concepts such as Generative Adversarial Networks (GANs), Variational Autoencoders (VAEs), and transformer models are essential for implementing GenAI solutions.
- **Cloud Services Overview**: Both AWS and Azure offer various AI and Machine Learning (ML) services tailored for GenAI applications, including pre-built models, training tools, and integration capabilities.

---

### Phase 2: Capability Assessment

**AWS**
- **Services**: Key offerings for GenAI include Amazon SageMaker, which simplifies building, training, and deploying ML models, and AWS Lambda for serverless computation.
- **Hardware Support**: AWS provides specialized hardware like EC2 P3 and P4 instances equipped with NVIDIA GPUs for training intense AI models, along with Inferentia chips optimized for inference.
- **Integration**: Seamless integration with a range of AWS services (data lakes, databases, analytics) offers a holistic environment for AI.

**Azure**
- **Services**: Azure AI and Machine Learning Studio provide extensive tools for building and managing AI workflows. Azure also offers services tailored towards businesses already embedded in the Microsoft ecosystem.
- **Hardware**: Utilizes FPGA and NVIDIA GPUs for processing power, facilitating high-performance GenAI applications.
- **Integration**: Strong integration with Microsoft services (such as Power BI, Office 365) enables enterprises invested in the Microsoft stack to leverage existing frameworks for AI.

---

### Phase 3: Performance and Benchmarking

**Benchmarking Approach:**
- Develop and deploy standardized GenAI tasks (e.g., image and text generation).
- Measure metrics such as training time, inference speed, and resource utilization on both platforms.

**Preliminary Results:**
- Expected variances in performance depending on hardware configurations.
- Typically, AWS's GPU-optimized instances show promising training times, while Azure is competitive in integration speed with Microsoft services.

---

### Phase 4: Cost Analysis

**Activities:**
- Gather cost data for compute instances, storage, and associated services.
- Simulate diverse usage scenarios (development, training, deployment).

**Cost Insights:**
- **AWS**: Variable costs based on service usage; potential hidden fees from data egress may affect total costs.
- **Azure**: Competitive pricing on bundled services may lower upfront costs, particularly for existing Microsoft customers.

---

### Phase 5: User Experience and Support

**User Feedback and Experience:**
- Developer feedback suggests AWS may have a steeper learning curve due to its extensive feature set, while Azure advocates praise its user-friendly interface.
- Robust documentation and support ecosystems exist for both platforms, but user preference may lean towards Azure for organizations familiar with Microsoft.

---

### Phase 6: Security and Compliance

**Security Measures:**
- Both platforms prioritize security with advanced encryption standards, identity management, and access control.
- Compliance certifications (GDPR, HIPAA) are comprehensive across both, but specific industry-focused offerings might be more prominent in Azure due to its enterprise focus.

---

### Conclusion and Deliverables

1. **Detailed Report**: A comprehensive report summarizing the comparative analysis, performance benchmarking, cost assessments, and user experiences.
2. **Presentation**: A condensed presentation summarizing critical findings and recommendations tailored for stakeholders.
3. **Supplementary Materials**: Benchmarking data, cost charts, and user feedback summaries to support decision-making.

---

### Timeline Overview
- **Literature Review**: 2 weeks
- **Capability Assessment**: 3 weeks
- **Performance Benchmarking**: 4 weeks
- **Cost Analysis**: 2 weeks
- **User Experience Assessment**: 3 weeks
- **Security Compliance Review**: 2 weeks
- **Final Report**: 2 weeks

---

### Budget Consideration
- Allocate funds for cloud service usage, tools for analysis, potential travel for user interviews, if necessary.

---

### Risks and Mitigations
- **Variability in Pricing**: Establish a regular review process for current pricing.
- **Potential Bias**: Ensure diverse sampling for feedback to maintain objectivity.
- **Rapidly Evolving Services**: Stay updated on service announcements and adjust the research as necessary.

---

This research serves as a foundational analysis for organizations looking to decide between AWS and Azure for their GenAI workloads, guiding them toward making informed strategic decisions.

╭─ gpt-4o-mini → gpt-4o ─╮
│ Quality: 8/10          │
│ Latency: 21.32s        │
│ Tokens: 2943           │
│ Score: 4.73            │
╰────────────────────────╯

### Comparative Analysis of AWS and Azure for Generative AI Workloads

#### 1. Introduction
Amazon Web Services (AWS) and Microsoft Azure are leading cloud service providers that offer extensive capabilities for running Generative AI workloads. The demand for scalable, cost-effective, and high-performance solutions has pushed both platforms to innovate and improve their services continuously.

#### 2. Research Findings

##### Services Offered
- **AWS**: Provides services like Amazon SageMaker for building, training, and deploying machine learning models. It also offers Amazon Bedrock for building GenAI capabilities, along with AI APIs like Amazon Comprehend.
- **Azure**: Features Azure Machine Learning for comprehensive model management and Azure OpenAI Service, which integrates OpenAI models like GPT-3 for robust generative capabilities.

##### Performance Metrics
- **Training Speed & Inference Latency**: Both AWS and Azure have optimized services for specific use-cases. AWS’s infrastructure, including its custom chips like Inferentia, often touts high performance for AI workloads. Azure, on the other hand, excels in integration with Microsoft's ecosystem, which can enhance performance for Microsoft-specific applications and AI models.
  
##### Cost Implications
- AWS and Azure both offer a variety of pricing models, including pay-as-you-go, reserved instances, and spot instances. Generally, the cost will vary based on specific resource requirements, but Azure is noted for competitive pricing in long-term commitments, while AWS offers more flexibility in short-duration workloads.

##### Ease of Use, Documentation, and Support
- **AWS**: Known for extensive documentation and a vast range of community resources. Its marketplace provides a plethora of third-party integrations.
- **Azure**: Offers seamless integration with Microsoft products, which can be beneficial for enterprises already using Microsoft technologies. Its documentation and tight integration with tools like Visual Studio are substantial advantages.

##### Security and Compliance
- Both AWS and Azure offer robust security and compliance features tailored to AI workloads. They include capabilities like encryption, identity management, and adherence to global standards and regulations such as GDPR.

#### 3. Conclusion
Both AWS and Azure provide strong capabilities for Generative AI, but the choice between them often boils down to specific organizational needs:
- **Choose AWS**: If your enterprise is already AWS-centric or if you require flexibility in short-term AI projects.
- **Choose Azure**: If your organization benefits from a tight integration with Microsoft products and aims to leverage OpenAI models extensively.

#### Recommendation
For companies seeking to deploy Generative AI solutions, a hybrid approach leveraging both AWS and Azure where applicable can offer the best of both worlds. Additionally, conducting a pilot test on both platforms with specific workloads can provide quantifiable insights into performance and cost-effectiveness to inform long-term decisions.

This comparative analysis underscores the versatility and capability of both AWS and Azure in supporting Generative AI workloads. Organizations are encouraged to evaluate their specific requirements and consider both strategic and cost-related factors when choosing a platform.

╭─ 🏆 Best Configuration ─╮
│ Planner: gpt-4o-mini    │
│ Summarizer: gpt-4o-mini │
│ Quality: 10             │
│ Latency: 28.38s         │
│ Score: 6.38             │
╰─────────────────────────╯